In [9]:
!pip install librosa numpy scikit-learn tensorflow soundfile matplotlib seaborn

<class 'OSError'>: Not available

In [8]:
import os
import numpy as np
import librosa
import librosa.display
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from tensorflow.keras import layers, models
warnings.filterwarnings('ignore')

print("✅ All libraries imported!")

<class 'ModuleNotFoundError'>: No module named 'librosa'

In [ ]:
class Config:
    SAMPLE_RATE = 16000
    TARGET_DURATION = 7
    N_MFCC = 13
    FRAME_LENGTH = 512
    HOP_LENGTH = 256
    BATCH_SIZE = 32
    EPOCHS = 50
    DROPOUT_RATE = 0.5
    
    CLASSES = ["Belly pain", "Burping", "Discomfort", "Hungry", "Tired"]
    
    @classmethod
    def get_target_length(cls):
        return cls.SAMPLE_RATE * cls.TARGET_DURATION

print("✅ Config loaded!")
print(f"Classes: {Config.CLASSES}")

In [ ]:
def extract_features(filepath):
    """
    استخراج ویژگی MFCC از فایل صوتی
    مطابق مقاله Section 2.3.3
    """
    try:
        # بارگذاری فایل صوتی
        signal, _ = librosa.load(filepath, sr=Config.SAMPLE_RATE)
        target_len = Config.get_target_length()
        
        # Padding/Trimming به طول 7 ثانیه
        if len(signal) < target_len:
            signal = np.pad(signal, (0, target_len - len(signal)))
        else:
            signal = signal[:target_len]
        
        # استخراج MFCC
        mfcc = librosa.feature.mfcc(
            y=signal,
            sr=Config.SAMPLE_RATE,
            n_mfcc=Config.N_MFCC,
            n_fft=Config.FRAME_LENGTH,
            hop_length=Config.HOP_LENGTH
        )
        
        # میانگین‌گیری در طول زمان (مطابق مقاله)
        return np.mean(mfcc, axis=1)
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return None

print("✅ Feature extraction ready!")

In [ ]:
def load_dataset(dataset_path="data"):
    """
    بارگذاری دیتاست Donate a Cry
    ساختار پوشه‌ها:
    data/
      ├── Belly pain/
      ├── Burping/
      ├── Discomfort/
      ├── Hungry/
      └── Tired/
    """
    X, y = [], []
    print("📂 Loading dataset...")
    
    for idx, class_name in enumerate(Config.CLASSES):
        class_path = os.path.join(dataset_path, class_name)
        
        # اگر پوشه وجود نداشت، می‌سازیم
        if not os.path.exists(class_path):
            os.makedirs(class_path, exist_ok=True)
            print(f"  ⚠️ Created directory: {class_path}")
            continue
        
        wav_files = glob.glob(os.path.join(class_path, "*.wav"))
        print(f"  {class_name}: {len(wav_files)} files")
        
        for wav in wav_files:
            features = extract_features(wav)
            if features is not None:
                X.append(features)
                y.append(idx)
    
    if len(X) == 0:
        print("\n⚠️ No data found! Creating dummy data for testing...")
        X, y = create_dummy_data()
    
    print(f"\n✅ Loaded {len(X)} samples with {X[0].shape[0]} features")
    return np.array(X), np.array(y)

def create_dummy_data():
    """ساخت داده‌های آزمایشی"""
    X, y = [], []
    for i in range(50):
        for class_idx in range(len(Config.CLASSES)):
            features = np.random.randn(Config.N_MFCC) * 0.5 + (class_idx * 0.3)
            X.append(features)
            y.append(class_idx)
    return np.array(X), np.array(y)

X, y = load_dataset()

In [ ]:
# نمایش توزیع کلاس‌ها
plt.figure(figsize=(10, 5))
sns.countplot(x=y)
plt.title("Class Distribution (Table 2 from paper)")
plt.xticks(range(len(Config.CLASSES)), Config.CLASSES, rotation=45)
plt.ylabel("Number of samples")
plt.xlabel("Class")
plt.tight_layout()
plt.show()

# نمایش آمار
for i, cls in enumerate(Config.CLASSES):
    count = np.sum(y == i)
    print(f"{cls:12s}: {count:3d} samples ({count/len(y)*100:.1f}%)")

In [ ]:
# تقسیم داده‌ها مطابق مقاله
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.60, stratify=y_temp, random_state=42
)

print("📊 Data Split:")
print(f"  Training:   {len(X_train):4d} ({len(X_train)/len(X)*100:5.1f}%)")
print(f"  Validation: {len(X_val):4d} ({len(X_val)/len(X)*100:5.1f}%)")
print(f"  Test:       {len(X_test):4d} ({len(X_test)/len(X)*100:5.1f}%)")
print(f"  Total:      {len(X):4d}")

In [ ]:
def create_model():
    """
    معماری ANN از مقاله - Figure 4
    5 لایه پنهان با Dropout 0.5
    """
    model = models.Sequential([
        # Layer 1: 128 units
        layers.Dense(128, activation='relu', input_shape=(Config.N_MFCC,)),
        layers.BatchNormalization(),
        layers.Dropout(Config.DROPOUT_RATE),
        
        # Layer 2: 128 units
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(Config.DROPOUT_RATE),
        
        # Layer 3: 64 units
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(Config.DROPOUT_RATE),
        
        # Layer 4: 64 units
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(Config.DROPOUT_RATE),
        
        # Layer 5: 32 units
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(Config.DROPOUT_RATE),
        
        # Output: 5 classes
        layers.Dense(len(Config.CLASSES), activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = create_model()
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6)
]

print("🚀 Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=Config.BATCH_SIZE,
    epochs=Config.EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ذخیره مدل
model.save("baby_cry_model.h5")
print("✅ Model saved as 'baby_cry_model.h5'")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# دقت
axes[0].plot(history.history['accuracy'], label='Training', color='blue')
axes[0].plot(history.history['val_accuracy'], label='Validation', color='orange')
axes[0].set_title('Learning Curves - Accuracy (Figure 6)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Training', color='blue')
axes[1].plot(history.history['val_loss'], label='Validation', color='orange')
axes[1].set_title('Learning Curves - Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# پیش‌بینی
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# محاسبه متریک‌ها
test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='weighted')

print("📊 Test Results:")
print(f"  Accuracy: {test_acc*100:.2f}%")
print(f"  F1-Score: {test_f1:.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=Config.CLASSES))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=Config.CLASSES,
    yticklabels=Config.CLASSES
)
plt.title('Confusion Matrix (Figure 5)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import files
import io

def test_with_file():
    print("📤 Upload an audio file to test...")
    uploaded = files.upload()
    
    for filename, content in uploaded.items():
        # ذخیره موقت
        with open(filename, 'wb') as f:
            f.write(content)
        
        # استخراج ویژگی
        features = extract_features(filename)
        if features is not None:
            # پیش‌بینی
            pred = model.predict(features.reshape(1, -1), verbose=0)
            pred_class = np.argmax(pred[0])
            confidence = np.max(pred[0])
            
            print(f"\n🎯 Result for {filename}:")
            print(f"  Class: {Config.CLASSES[pred_class]}")
            print(f"  Confidence: {confidence*100:.1f}%")
            
            print("\n  All probabilities:")
            for i, cls in enumerate(Config.CLASSES):
                bar = '█' * int(pred[0][i] * 30)
                print(f"    {cls:12s}: {pred[0][i]*100:5.1f}% {bar}")
        else:
            print(f"❌ Failed to process {filename}")

# اجرا
test_with_file()

In [7]:
# =====================================================
# CELL 14: Simple Test with Audio File (Manual Path)
# =====================================================

def test_with_file():
    """
    تست با فایل صوتی - وارد کردن مسیر دستی
    """
    print("\n📂 Enter the path to your audio file:")
    print("  Example: data/Hungry/sample.wav")
    print("  Or: /home/user/audio.wav")
    print("  Or: C:\\Users\\audio.wav")
    print("  Type 'skip' to skip")
    
    filepath = input("File path: ").strip().strip('"').strip("'")
    
    if filepath.lower() == 'skip':
        print("⏭️ Skipped!")
        return
    
    if not os.path.exists(filepath):
        print(f"❌ File not found: {filepath}")
        return
    
    try:
        # بارگذاری فایل
        print(f"📄 Processing: {filepath}")
        signal, _ = librosa.load(filepath, sr=Config.SAMPLE_RATE)
        
        # استخراج ویژگی
        features = extract_features_from_signal(signal)
        
        if features is not None:
            # پیش‌بینی
            pred = model.predict(features.reshape(1, -1), verbose=0)
            pred_class = np.argmax(pred[0])
            confidence = np.max(pred[0])
            
            print(f"\n🎯 Result for {os.path.basename(filepath)}:")
            print(f"  Class: {Config.CLASSES[pred_class]}")
            print(f"  Confidence: {confidence*100:.1f}%")
            
            print("\n  All probabilities:")
            for i, cls in enumerate(Config.CLASSES):
                bar = '█' * int(pred[0][i] * 30)
                print(f"    {cls:12s}: {pred[0][i]*100:5.1f}% {bar}")
                
            # نمایش شکل موج
            plt.figure(figsize=(10, 3))
            plt.plot(signal[:5000], color='#ff69b4', alpha=0.7)
            plt.title(f"Audio Waveform: {os.path.basename(filepath)}")
            plt.xlabel("Samples")
            plt.ylabel("Amplitude")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print(f"❌ Failed to extract features")
            
    except Exception as e:
        print(f"❌ Error: {e}")

# اجرا
test_with_file()


📂 Enter the path to your audio file:
  Example: data/Hungry/sample.wav
  Or: /home/user/audio.wav
  Or: C:\Users\audio.wav
  Type 'skip' to skip


File path:  skip


⏭️ Skipped!
